In [ ]:
!pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
torch.manual_seed(42)

In [ ]:
#I loaded this with AI
import torchvision
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Download and load the training data
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)

# Download and load the test data
test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)


cpu


In [ ]:
# Extract images and labels from the PyTorch dataset
# Flatten the 28x28 images into 784 features
X_raw = train_data.data.numpy().reshape(-1, 28*28)
y_raw = train_data.targets.numpy()
X_test_raw = test_data.data.numpy().reshape(-1, 28*28)
y_test_raw = test_data.targets.numpy()

data = pd.DataFrame(X_test_raw)
data.insert(0, 'label', y_test_raw)

# Create a DataFrame
df = pd.DataFrame(X_raw)

# Insert the label as the first column to match your previous setup
df.insert(0, 'label', y_raw)
df


,label,0,1,2,3,4,5,6,7,8,...,774,775,776,777,778,779,780,781,782,783
0,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,119,114,130,76,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,33,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,66,54,50,5,0,1,0,0,0,0


In [ ]:
X_train = df.iloc[:,1:].values/255.0
y_train = df.iloc[:,0].values

X_test = data.iloc[:,1:].values/255.0
y_test = data.iloc[:,0].values

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, feature, label):
    self.feature = torch.tensor(feature, dtype = torch.float32)
    self.label = torch.tensor(label, dtype = torch.long)

  def __len__(self):
    return len(self.feature)

  def __getitem__(self, idx):
    return self.feature[idx], self.label[idx]

In [ ]:
#object of custom dataset that we had created above
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [ ]:
class MyNN(nn.Module):

  def __init__(self,input_dim, output_dim, num_hidden_layer, neuron_per_layer, dropout_rate):
    super().__init__()

    layer = []

    for i in range(num_hidden_layer):

      layer.append(nn.Linear(input_dim, neuron_per_layer))
      layer.append(nn.BatchNorm1d(neuron_per_layer))
      layer.append(nn.ReLU())
      layer.append(nn.Dropout(dropout_rate))
      input_dim = neuron_per_layer

    layer.append(nn.Linear(input_dim, output_dim))

  #Now we will make a model
    self.model = nn.Sequential(*layer)

  def forward(self,x):

    return self.model(x)


In [ ]:
#study for optuna
study = optuna.create_study(direction = 'maximize')

[I 2026-04-26 05:52:51,377] A new study created in memory with name: no-name-1412d6a7-9fc7-4284-96fb-15306c09c388


In [ ]:
def objective(trial):

  # Next hyperparameter values from the space space
  num_hidden_layer = trial.suggest_int("num_hidden_layer", 1, 5)
  neuron_per_layer = trial.suggest_int("neuron_per_layer", 8, 128, step = 8)
  epochs = trial.suggest_int("epochs", 10,50,step = 10)
  learning_rate = trial.suggest_float("learning_rate",1e-5, 1e-1, log = True)
  dropout_rate = trial.suggest_float("dropout_rate", 0,0.5,step = 0.1)
  batch_size = trial.suggest_categorical("batch_size", [16,32,64,128])
  optimizer = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
  weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3,log = True)
  # Model init
  input_dim = 784
  output_dim = 10
  model = MyNN(input_dim,output_dim,num_hidden_layer, neuron_per_layer, dropout_rate)
  model.to(device)
  #train test loader
  train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
  test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)


  # optimizer select
  criterian = nn.CrossEntropyLoss()


  if optimizer == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr= learning_rate, weight_decay = weight_decay)

  elif optimizer =='SGD':
    optimizer = optim.SGD(model.parameters(), lr= learning_rate, weight_decay = weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr= learning_rate, weight_decay = weight_decay)

  # Training loop
  for epoch in range(epochs):
   # total_epoch_loss = 0
    for batch_feature, batch_label in train_loader:
      batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)

      #forward
      optimizer.zero_grad()
      output = model.forward(batch_feature)

      #loss calculation
      loss = criterian(output, batch_label)
     # total_epoch_loss = total_epoch_loss + loss.item()
      #back propogation
      loss.backward()
      #update
      optimizer.step()
    #avg = total_epoch_loss/len(train_loader)
    #print(f"Epoch : {epoch + 1}/{epochs}\tLoss : {avg}")

  model.eval()

  # Evaluation
  total = 0
  correct = 0

  with torch.no_grad():
    for batch_feature, batch_label in test_loader:
      batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)

      output = model(batch_feature)

      _, predicted = torch.max(output, 1)
      total = total + batch_label.shape[0]
      correct = correct + (predicted == batch_label).sum().item()
  accuracy = correct/total

  return accuracy




In [ ]:
study.optimize(objective, n_trials = 10)

[I 2026-04-26 05:59:50,246] Trial 3 finished with value: 0.8863 and parameters: {'num_hidden_layer': 4, 'neuron_per_layer': 80, 'epochs': 30, 'learning_rate': 4.419224192259332e-05, 'dropout_rate': 0.0, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 7.34621758144589e-05}. Best is trial 3 with value: 0.8863.
[I 2026-04-26 06:00:24,537] Trial 4 finished with value: 0.8791 and parameters: {'num_hidden_layer': 1, 'neuron_per_layer': 128, 'epochs': 30, 'learning_rate': 0.006106298497959368, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 1.102584816549035e-05}. Best is trial 3 with value: 0.8863.
[I 2026-04-26 06:01:52,896] Trial 5 finished with value: 0.8925 and parameters: {'num_hidden_layer': 4, 'neuron_per_layer': 120, 'epochs': 30, 'learning_rate': 6.327946581139984e-05, 'dropout_rate': 0.1, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.4730430480212996e-05}. Best is trial 5 with value: 0.8925.
[I 2026-04-26 06:02:56,520] Trial 6 finishe

In [ ]:
study.best_value

0.8925

In [ ]:
study.best_params

{'num_hidden_layer': 4,
 'neuron_per_layer': 120,
 'epochs': 30,
 'learning_rate': 6.327946581139984e-05,
 'dropout_rate': 0.1,
 'batch_size': 64,
 'optimizer': 'Adam',
 'weight_decay': 1.4730430480212996e-05}